# 4주차 팀과제 — 14조 · 미션 D (방과후 청소년 공간)

**의사결정자**: 청소년정책 담당자 · **결정할 것**: 방과후 청소년이 많이 머무는 자치구에 공간 신설
**범위**: 15~18시 · 10대 · **절대**: 10대 합계 · **비중**: 같은 시간대 전 연령대 대비

## STEP① 질문 정의 — 가설 문장 (초안담당자: ________ / 15점)

**가설**: 오후 방과후 시간대(15~18시)의 10대 생활인구는 거주지가 아니라 학원·학교 등 사교육/교육시설이 밀집한 자치구에 몰릴 것이다. 따라서 10대 절대 합계 상위 3개 자치구에는 서울의 대표적 학원가로 알려진 강남구·양천구·노원구가 다수 포함될 것이다. 다만 비중(같은 시간대 전 연령대 대비 10대 비율) 상위 자치구는, 강남구·송파구처럼 업무·상업 유동인구까지 많아 비중이 희석되는 번화한 학원가보다, 학원가이면서 동시에 전통적으로 주거 비중이 높은(이른바 '베드타운') 노원구처럼 다른 연령대 유동인구가 상대적으로 적은 자치구에서 더 두드러질 것이다.

> 위 자치구 선정 근거(학원 수·주간인구지수 등 외부 자료)는 `00_가설_질문정의.md` 3-1절 참고.

**반대 결과가 나와도 받아들일 수 있는가(1줄 필수)**: 받아들인다 — 절대·비중 상위 자치구가 위 학원가 목록(강남·양천·노원)과 무관하게 나오면 '학원가 밀집 가설'을 기각하고 공원·대형 상업시설·환승거점·산업단지 통근 인구 등 비학업적 체류 요인으로 설명을 바꾸며, 이 데이터가 2026-04-15 단 하루·교육용으로 오류가 삽입된 가공 데이터라는 한계도 결론에 함께 명시한다.


In [ ]:
# 0. 환경 준비
!pip install -q koreanize-matplotlib
import koreanize_matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


## STEP② 데이터 이해 (초안담당자: ________ / 10점)

`docs/수집_기록표.md`의 수집 기록표·컬럼 5개 측정 척도를 참고하세요.


In [ ]:
import os

# Colab(파일을 노트북과 같은 위치에 업로드)과 로컬(data/ 폴더에 원본 보존) 경로가 달라서 둘 다 시도
_candidates = [
    '서울시_생활인구_202604_dl0830.csv',
    '../data/서울시_생활인구_202604_dl0830.csv',
    'data/서울시_생활인구_202604_dl0830.csv',
]
_path = next(p for p in _candidates if os.path.exists(p))

df = pd.read_csv(_path, encoding='cp949')
df.columns = [c.strip() for c in df.columns]
print('불러온 경로:', _path)
print('행, 열:', df.shape)
df.head(10)


In [ ]:
# 컬럼별로 값을 살펴보면서 자료구조 파악 (측정 척도 판단용)

print('[기준일자] 고유값:', df['기준일자'].unique())

print('\n[자치구] 값 개수:', df['자치구'].nunique())
print(sorted(df['자치구'].unique()))

print('\n[시간대] 범위:', df['시간대'].min(), '~', df['시간대'].max(), '| dtype:', df['시간대'].dtype)

print('\n[연령대] 값 목록:', df['연령대'].unique())

print('\n[생활인구수] dtype:', df['생활인구수'].dtype)
_comma_example = df.loc[df['생활인구수'].astype(str).str.contains(','), '생활인구수'].head(2).tolist()
_has_dash = (df['생활인구수'] == '-').any()
print('콤마 포함 예시:', _comma_example, "| '-' 표시 존재 여부:", _has_dash)


## STEP③ 탐색 — 첫인상 확인 + 이상 징후 3개 이상 + 정제 전 1차 집계 (초안담당자: ________ / 15점)

AI에게 시킬 프롬프트 예시: "이 CSV의 첫 10줄, 행·열 개수, 각 컬럼의 자료형과 결측 개수를 보여줘. 이상해 보이는 값도 함께 지적해줘."


In [ ]:
# 자료형 · 결측 개수
print(df.dtypes)
print()
print(df.isnull().sum())


In [ ]:
# 이상 징후 기록 (최소 3개 — docs/data_description.md 참고)
# 1)
# 2)
# 3)


In [ ]:
# 정제 전 1차 집계 — 우리 미션 범위(15~18시 · 10대)만 먼저 필터링해서 행 수를 확인
before = df[(df['시간대'].between(15, 18)) & (df['연령대'].str.strip() == '10대')]
print('정제 전 미션 범위 행 수:', len(before))
before.head()


## STEP④ 정제 — 정제 로그 5줄 이상 (처리·건수·근거·영향) (초안담당자: ________ / 25점, 가장 큰 배점)

`logs/정제로그.md` 초안을 참고해 팀이 실제로 결정한 내용으로 코드를 작성하세요.
**몇 건을 지웠는지가 아니라 '왜' 지웠는지/안 지웠는지가 점수입니다.**


In [ ]:
clean = df.copy()  # 원본 보존

# 1) 컬럼명 공백은 이미 위에서 strip 처리함

# 2) 생활인구수: 콤마 제거 후 숫자형 변환
clean['생활인구수'] = pd.to_numeric(
    clean['생활인구수'].astype(str).str.replace(',', ''), errors='coerce')

# 3) ~ 6) 결측('-' 포함)·음수·이상치·중복 처리는 팀 판단에 따라 작성
#    (판단 근거는 logs/정제로그.md에 반드시 기록)

print('정제 전:', len(df), '행 → 정제 후:', len(clean), '행')


## STEP⑤ 결론 — 절대·비중 교차 + 권고·근거·한계 (초안담당자: ________ / 20점)

AI 프롬프트 예시:
- [절대] "15~18시·10대만 골라서 자치구별 합계를 구하고, 상위 5개를 표로 보여줘. 필터 전후 행 수도 함께 출력해줘."
- [비중] "같은 시간대 전 연령 대비 10대의 비중을 자치구별로 계산해 상위 5개를 보여줘. 절대 순위와 겹치는 구도 알려줘."


In [ ]:
# 절대 — 15~18시 · 10대 자치구별 합계 상위 5
mission = clean[(clean['시간대'].between(15, 18)) & (clean['연령대'].str.strip() == '10대')]
print('필터 전:', len(clean), '행 → 필터 후:', len(mission), '행')
absolute_top5 = mission.groupby('자치구')['생활인구수'].sum().sort_values(ascending=False).head(5)
absolute_top5


In [ ]:
# 비중 — 같은 시간대 전 연령대 대비 10대 비중 자치구별 상위 5
same_time = clean[clean['시간대'].between(15, 18)]
total_by_gu = same_time.groupby('자치구')['생활인구수'].sum()
teen_by_gu = mission.groupby('자치구')['생활인구수'].sum()
share = (teen_by_gu / total_by_gu * 100).sort_values(ascending=False).head(5)
share


### 결론 3요소
- **권고**: (신규 공간 후보 3곳, 또는 2곳+판단보류 1곳)
- **근거**: (절대·비중 교차 결과)
- **한계**: (하루치·가공 데이터, 생활인구≠거주인구 등)
